In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

In [2]:
class TextDataset:
    def __init__(self, file_path='the little prince.txt', tokenize=True):
        if isinstance(file_path, bool):
            tokenize = file_path
            file_path = 'the little prince.txt'

        # 利用nltk函数进行分句和分词
        with open(file_path, 'r', encoding='utf-8') as file:
            text = file.read()
        if tokenize:
            self.sentences = sent_tokenize(text.lower())
            self.tokens = [word_tokenize(sent) for sent in self.sentences]
        else:
            self.text = text

    def build_vocab(self, min_freq=1):
        # 统计词频
        frequency = defaultdict(int)
        for sentence in self.tokens:
            for token in sentence:
                frequency[token] += 1
        self.frequency = frequency

        # 加入<unk>处理未登录词，加入<pad>用于对齐变长输入进而加速
        self.token2id = {'<unk>': 1, '<pad>': 0}
        self.id2token = {1: '<unk>', 0: '<pad>'}
        for token, freq in sorted(frequency.items(), key=lambda x: -x[1]):
            # 丢弃低频词
            if freq > min_freq:
                self.token2id[token] = len(self.token2id)
                self.id2token[len(self.id2token)] = token
            else:
                break

    def get_word_distribution(self):
        distribution = np.zeros(vocab_size)
        for token, freq in self.frequency.items():
            if token in dataset.token2id:
                distribution[dataset.token2id[token]] = freq
            else:
                # 不在词表中的词按<unk>计算
                distribution[1] += freq
        distribution /= distribution.sum()
        return distribution

    # 将分词结果转化为索引表示
    def convert_tokens_to_ids(self, drop_single_word=True):
        self.token_ids = []
        for sentence in self.tokens:
            token_ids = [self.token2id.get(token, 1) for token in sentence]
            # 忽略只有一个token的序列，无法计算loss
            if len(token_ids) == 1 and drop_single_word:
                continue
            self.token_ids.append(token_ids)
        
        return self.token_ids


In [3]:
from collections import defaultdict
import re
import numpy as np

try:
    from nltk import sent_tokenize as _nltk_sent_tokenize
    from nltk import word_tokenize as _nltk_word_tokenize

    _nltk_sent_tokenize("test sentence.")
    _nltk_word_tokenize("test sentence.")
    sent_tokenize = _nltk_sent_tokenize
    word_tokenize = _nltk_word_tokenize
    print("使用 NLTK 分词")
except LookupError:
    print("使用本地正则分词")

    def sent_tokenize(text):
        return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]

    def word_tokenize(sentence):
        return re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?|[0-9]+|[^\w\s]", sentence)


dataset = TextDataset(file_path=r"D:\DeepLearning\DataSet\Text\anna.txt")

使用本地正则分词


接下来建立词表，截断过长的序列，将序列填充（padding）到相同长度（不使用词嵌入）

In [4]:
import numpy as np

dataset.build_vocab()
sent_tokens = dataset.convert_tokens_to_ids()
# 截断和填充
max_len=40
for i, tokens in enumerate(sent_tokens):
    tokens = tokens[:max_len]
    tokens += [dataset.token2id['<pad>']] * (max_len - len(tokens))
    sent_tokens[i] = tokens
    

In [5]:
import torch
from torch import nn
import torch.nn.functional as F

from torch.utils.data import DataLoader
from torch.optim import SGD, Adam
import numpy as np
from tqdm import tqdm, trange


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"当前使用设备: {device}")

当前使用设备: cuda


In [7]:
def normal(shape):
    return torch.randn(size=shape) * 0.01

class RNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(RNN, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        # 将输入与隐状态分别经过线性变化后相加
        self.W_xh = nn.Parameter(normal((input_size, hidden_size)))
        self.W_hh = nn.Parameter(normal((hidden_size, hidden_size)))
        self.b_h = nn.Parameter(torch.zeros(hidden_size))
    
    def init_rnn_state(self, batch_size, hidden_size):
        return (torch.zeros((batch_size, hidden_size), dtype=torch.float, device=self.W_xh.device),)
    
    def forward(self, inputs, states):
        seq_len, batch_size, _ = inputs.shape
        hidden_state, = states
        hiddens = []
        for step in range(seq_len):
            # 输入hidden_state与inputs经过线性变换后相加，
            # 输出的hidden_state也是下一时刻输入的hidden_state
            xh = torch.mm(inputs[step], self.W_xh)
            hh = torch.mm(hidden_state, self.W_hh)
            hidden_state = xh + hh + self.b_h
            hidden_state = torch.tanh(hidden_state)
            hiddens.append(hidden_state)
        # 返回所有时刻的hidden_state: seq_len * batch_size * hidden_size
        # 以及最后时刻的hidden_state 
        # batch_size * hidden_size
        return torch.stack(hiddens, dim=0), (hidden_state,)

# 在循环神经网络的基础上添加语言模型的输入输出、损失计算等
class RNNLM(nn.Module):
    def __init__(self, model, vocab_size, hidden_size):
        super(RNNLM, self).__init__()
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.model = model
        self.W_hq = nn.Parameter(normal((hidden_size, vocab_size)))
        self.b_q = nn.Parameter(torch.zeros(vocab_size))
        
    def forward(self, input_ids):
        batch_size, seq_len = input_ids.shape
        # input_ids形状为batch_size * seq_len，翻转为seq_len * batch_size，
        # 将seq_len放在第一维方便计算
        input_ids = torch.permute(input_ids, (1, 0))
        # seq_len * batch_size * embed_size
        embed = self.embedding(input_ids)
        # batch_size * hidden_size
        states = self.model.init_rnn_state(batch_size, self.hidden_size)
        hiddens, _ = self.model(embed, states)
    
        hiddens = torch.flatten(hiddens[:-1], start_dim=0, end_dim=1)
        output_states = torch.mm(hiddens, self.W_hq) + self.b_q
        labels = torch.flatten(input_ids[1:], start_dim=0, end_dim=1)
        loss_fct = nn.CrossEntropyLoss(ignore_index=0)
        loss = loss_fct(output_states, labels)
        return loss


In [ ]:
# 梯度裁剪
def grad_clipping(model, theta=1):
    params = [p for p in model.parameters() if p.requires_grad and p.grad is not None]
    if not params:
        return
    norm = torch.sqrt(sum(torch.sum(p.grad ** 2) for p in params))
    if norm > theta:
        for param in params:
            param.grad[:] *= theta / norm


def train_rnn_lm(data_loader, rnn, vocab_size, hidden_size=128,
                 epochs=200, learning_rate=1e-3):
    # 准备模型、优化器等
    rnn_lm = RNNLM(rnn, vocab_size, hidden_size).to(device)
    optimizer = Adam(rnn_lm.parameters(), lr=learning_rate)
    rnn_lm.zero_grad()
    rnn_lm.train()

    epoch_loss = []
    with trange(epochs, desc='epoch', ncols=60) as pbar:
        for epoch in pbar:
            for step, batch in enumerate(data_loader):
                batch = batch.to(device)
                loss = rnn_lm(batch)
                pbar.set_description(f'epoch: {epoch}, ' +
                    f'loss={loss.item():.4f}')
                loss.backward()
                grad_clipping(rnn_lm)
                optimizer.step()
                rnn_lm.zero_grad()
            epoch_loss.append(loss.item())

    epoch_loss = np.array(epoch_loss)
    # 打印损失曲线
    plt.plot(range(len(epoch_loss)), epoch_loss)
    plt.xlabel('training epoch')
    plt.ylabel('loss')
    plt.show()


下面实现一个带有缩放点乘注意力的循环神经网络，并用其训练语言模型。

In [9]:
class AttentionRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(AttentionRNN, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.W_xh = nn.Parameter(normal((input_size, hidden_size)))
        self.W_hh = nn.Parameter(normal((hidden_size, hidden_size)))
        self.b_h = nn.Parameter(torch.zeros(hidden_size))
    
    def init_rnn_state(self, batch_size, hidden_size):
        device = self.W_xh.device
        dtype = self.W_xh.dtype
        return (
            torch.zeros((batch_size, hidden_size), dtype=dtype, device=device),
            torch.zeros((batch_size, hidden_size), dtype=dtype, device=device),
        )
    
    def attention(self, query, keys, values):
        query = torch.unsqueeze(query, 1)
        keys = torch.permute(keys, (0, 2, 1))
        attention_scores = torch.bmm(query, keys) / torch.sqrt(torch.tensor(self.hidden_size, dtype=query.dtype, device=query.device))
        attention_weights = F.softmax(attention_scores, dim=2)   # 注意 dim=2 或 dim=1？根据原始代码 dim=1，但 query 形状为 batch*1*prev_len，softmax 应在最后一维（prev_len）上，即 dim=2。原始写 dim=1 是错的，需要修正。
        attention_state = torch.squeeze(torch.bmm(attention_weights, values))
        return attention_state

    def forward(self, inputs, states):
        seq_len, batch_size, _ = inputs.shape
        hidden_state, _ = states     
        
        hiddens = []
        attention_hiddens = []             
        attention_state = hidden_state     # 保底初始值
        
        for step in range(seq_len):
            xh = torch.mm(inputs[step], self.W_xh)
            hh = torch.mm(hidden_state, self.W_hh)
            hidden_state = xh + hh + self.b_h
            hidden_state = torch.tanh(hidden_state)
            
            if step > 0:
                query = hidden_state
                keys = values = torch.permute(torch.stack(hiddens, dim=0), (1, 0, 2))
                attention_state = self.attention(query, keys, values)
                attention_hiddens.append(attention_state)
            else:
                # 第0步没有历史，直接用当前隐状态
                attention_hiddens.append(hidden_state)
            
            hiddens.append(hidden_state)
        
        # 返回所有时刻的注意力状态（列表 -> 张量），以及最后一步的 attention_state
        return torch.stack(attention_hiddens, dim=0), (attention_state,)

In [10]:
data_loader = DataLoader(torch.tensor(sent_tokens, dtype=torch.long, device=device),
    batch_size=16, shuffle=True)

sent_tokens = np.array(sent_tokens)
vocab_size = len(dataset.token2id)

In [11]:
attention_rnn = AttentionRNN(128, 128).to(device)

train_rnn_lm(data_loader, attention_rnn, vocab_size, hidden_size=128, 
    epochs=200, learning_rate=1e-3)

epoch-1, loss=5.5678:   0%| | 1/200 [00:48<2:39:34, 48.11s/i


KeyboardInterrupt: 

**多头注意力**

In [12]:
# 多头注意力循环神经网络
class MultiHeadAttentionRNN(AttentionRNN):
    def __init__(self, input_size, hidden_size, num_heads=4):
        super().__init__(input_size, hidden_size)
        # 简单起见，一般要求hidden_size能够被num_heads整除
        assert hidden_size % num_heads == 0
        self.num_heads = num_heads
        # 多头注意力参数，用于将查询、键、值映射到子空间
        self.W_aq = nn.Parameter(normal((hidden_size, hidden_size)))
        self.b_aq = nn.Parameter(torch.zeros(hidden_size))
        self.W_ak = nn.Parameter(normal((hidden_size, hidden_size)))
        self.b_ak = nn.Parameter(torch.zeros(hidden_size))
        self.W_av = nn.Parameter(normal((hidden_size, hidden_size)))
        self.b_av = nn.Parameter(torch.zeros(hidden_size))
        self.W_ac = nn.Parameter(normal((hidden_size, hidden_size)))
        self.b_ac = nn.Parameter(torch.zeros(hidden_size))

    # 多头缩放点乘注意力
    def attention(self, query, keys, values):
        """
        query: batch_size * hidden_size
        keys/values: batch_size * prev_len * hidden_size
        """
        query = torch.mm(query, self.W_aq) + self.b_aq
        ori_shape = keys.size()
        
        keys = torch.reshape(torch.mm(torch.flatten(keys, 
                start_dim=0, end_dim=1), self.W_ak) + 
                self.b_ak, ori_shape)
        values = torch.reshape(torch.mm(torch.flatten(values, 
                start_dim=0, end_dim=1), self.W_av) + 
                self.b_av, ori_shape)
        # batch_size * 1 * hidden_size
        query = torch.unsqueeze(query, 1)
        # batch_size * hidden_size * prev_len
        keys = torch.permute(keys, (0, 2, 1))
        
        head_size = self.hidden_size // self.num_heads
        query = torch.split(query, head_size, 2)
        keys = torch.split(keys, head_size, 1)
        values = torch.split(values, head_size, 2)
        
        heads = []
        for i in range(self.num_heads):
            # batch_size * 1 * prev_len
            head_scores = torch.bmm(query[i], keys[i]) / np.sqrt(
                self.hidden_size // self.num_heads) 
            # batch_size * 1 * prev_len
            head_weights = F.softmax(head_scores, dim=1)
            # batch_size * head_size
            head_state = torch.squeeze(torch.bmm(head_weights, 
                values[i])) 
            heads.append(head_state)
        heads = torch.cat(heads, dim=1)        
        attention_state = torch.mm(heads, self.W_ac) + self.b_ac

        return attention_state

data_loader = DataLoader(torch.tensor(sent_tokens, dtype=torch.long, device=device),
    batch_size=16, shuffle=True)


In [13]:
mha_rnn = MultiHeadAttentionRNN(128, 128).to(device)

train_rnn_lm(data_loader, mha_rnn, vocab_size, hidden_size=128, 
    epochs=200, learning_rate=1e-3)

epoch-0, loss=6.9059:   0%|         | 0/200 [00:12<?, ?it/s]


KeyboardInterrupt: 